# 01 · Data Preparation
**Medical Chatbot MVP** - dataset preparation and validation

This notebook:
1. Loads and inspects raw data
2. Validates structure and coverage
3. Builds documents for RAG indexing
4. Saves processed data
5. Generates evaluation dataset (questions + golden answers)


In [ ]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime
from collections import Counter

DATA_DIR = Path('../data')
RAW_DIR = DATA_DIR / 'raw'
KB_DIR = DATA_DIR / 'processed' / 'knowledge_base'
KB_DIR.mkdir(parents=True, exist_ok=True)

print('Setup OK')

In [ ]:
with open(RAW_DIR / 'doctors.json', encoding='utf-8') as f:
    doctors = json.load(f)

with open(RAW_DIR / 'specialties.json', encoding='utf-8') as f:
    specialties = json.load(f)

with open(RAW_DIR / 'appointments.json', encoding='utf-8') as f:
    appointments = json.load(f)

df_doctors = pd.DataFrame(doctors)
df_specs = pd.DataFrame(specialties)

print(f'Doctors: {len(doctors)}')
print(f'Specialties: {len(specialties)}')
print(f'Appointments: {len(appointments)}')
print()
df_doctors[['id','name','specialty','years_experience','rating','accepts_nhif']].head(10)

In [ ]:
required_fields = ['id','name','specialty_id','specialty','bio','expertise',
                    'languages','price_consultation','rating','room']

print('=== Doctor validation ===')
issues = []
for doc in doctors:
    for field in required_fields:
        if field not in doc or doc[field] is None:
            issues.append(f"{doc['id']} — missing field '{field}'")

if issues:
    print('ISSUES FOUND:')
    for i in issues: print(i)
else:
    print('All required fields present')

spec_ids = {s['id'] for s in specialties}
doctor_spec_ids = {d['specialty_id'] for d in doctors}
missing_specs = doctor_spec_ids - spec_ids

if missing_specs:
    print(f'ERROR: Unknown specialty_id values: {missing_specs}')
else:
    print('All specialty_id values are valid')

doctor_ids = {d['id'] for d in doctors}
appt_ids = set(appointments.keys())
no_appt = doctor_ids - appt_ids

if no_appt:
    print(f'WARNING: Doctors without appointment slots: {no_appt}')
else:
    print('All doctors have appointment slots')

## Dataset distribution

In [ ]:
# Doctors per specialty
spec_counts = df_doctors['specialty'].value_counts().reset_index()
spec_counts.columns = ['specialty', 'count']

fig = px.bar(spec_counts, x='count', y='specialty', orientation='h',
             title='Doctors per specialty',
             color='count', color_continuous_scale='Blues')
fig.update_layout(showlegend=False, yaxis={'categoryorder':'total ascending'})
fig.show()

print(f"Avg consultation price: {df_doctors['price_consultation'].mean():.0f} BGN")
print(f"Avg rating: {df_doctors['rating'].mean():.2f}")
print(f"Doctors accepting NHIF: {df_doctors['accepts_nhif'].sum()} of {len(df_doctors)}")

In [ ]:
# Symptom coverage per specialty
all_symptoms = []
for spec in specialties:
    for sym in spec['symptoms']:
        all_symptoms.append({'specialty': spec['specialty'], 'symptom': sym})

df_sym = pd.DataFrame(all_symptoms)
print(f'Total symptom entries: {len(df_sym)}')
print(f'Unique symptoms: {df_sym["symptom"].nunique()}')
print()
print('Symptoms per specialty:')
print(df_sym.groupby('specialty')['symptom'].count().sort_values(ascending=False).to_string())

In [ ]:
# Appointment slot availability
stats = []
for doc_id, data in appointments.items():
    total = len(data['slots'])
    available = sum(1 for s in data['slots'] if s['available'])
    stats.append({
        'id': doc_id,
        'name': data['doctor_name'],
        'total': total,
        'available': available,
        'booked': total - available,
        'availability_pct': round(available / total * 100) if total > 0 else 0
    })

df_appt = pd.DataFrame(stats)
print(f'Total slots: {df_appt["total"].sum()}')
print(f'Available: {df_appt["available"].sum()}')
print(f'Booked: {df_appt["booked"].sum()}')
print(f'Avg occupancy: {100 - df_appt["availability_pct"].mean():.0f}%')
df_appt[['name','total','available','availability_pct']].head(10)

## Build RAG documents

Transform structured JSON data into natural language documents suitable for embedding and retrieval.

In [ ]:
def doctor_to_document(doc: dict) -> dict:
    """
    Convert a doctor JSON record into a text document for RAG.
    Returns a dict with 'text' (for embedding) and 'metadata' (for filtering).
    The document text is intentionally kept in Bulgarian — the language of the data.
    """
    nhif_text = 'Приема по НЗОК с направление' if doc.get('accepts_nhif') else 'Само платени прегледи'
    academic = f"{doc['academic_title']} " if doc.get('academic_title') else ''
    expertise = ', '.join(doc.get('expertise', []))
    languages = ', '.join(doc.get('languages', ['български']))

    text = f"""Лекар: {academic}{doc['name']}
Специалност: {doc['specialty']}
Опит: {doc['years_experience']} години
Образование: {doc.get('education', '')}

{doc.get('bio', '')}

Специализира в: {expertise}
Езици: {languages}
Цена на преглед: {doc['price_consultation']} лева
{nhif_text}
Оценка: {doc['rating']}/5.0 ({doc.get('reviews_count', 0)} отзива)
Кабинет: {doc.get('room', 'N/A')}"""

    return {
        'id': f"doctor_{doc['id']}",
        'text': text,
        'metadata': {
            'type': 'doctor',
            'doctor_id': doc['id'],
            'specialty_id': doc['specialty_id'],
            'specialty': doc['specialty'],
            'accepts_nhif': doc.get('accepts_nhif', False),
            'price': doc['price_consultation'],
            'rating': doc['rating'],
        }
    }

doctor_docs = [doctor_to_document(d) for d in doctors]
print(f'Generated {len(doctor_docs)} doctor documents')
print()
print('--- Sample (dr_001) ---')
print(doctor_docs[0]['text'])

In [ ]:
def specialty_to_document(spec: dict) -> dict:
    """
    Convert a specialty record into a document mapping symptoms to specialist.
    This is the core of the routing logic.
    Document text is in Bulgarian to match user queries.
    """
    symptoms_text = ', '.join(spec['symptoms'])
    urgent_text = ', '.join(spec.get('urgent_symptoms', []))
    procedures_text = ', '.join(spec.get('common_procedures', []))

    text = f"""Специалност: {spec['specialty']}
{spec['description']}

При кои симптоми трябва да се запише час при {spec['specialty']}:
{symptoms_text}

СПЕШНИ симптоми — незабавно потърсете помощ (112 или Спешно отделение):
{urgent_text}

Типични изследвания и процедури:
{procedures_text}

Продължителност на преглед: около {spec['avg_consultation_minutes']} минути"""

    return {
        'id': f"specialty_{spec['id']}",
        'text': text,
        'metadata': {
            'type': 'specialty',
            'specialty_id': spec['id'],
            'specialty': spec['specialty'],
        }
    }

specialty_docs = [specialty_to_document(s) for s in specialties]
print(f'Generated {len(specialty_docs)} specialty documents')
print()
print('--- Sample (cardiology) ---')
print(specialty_docs[0]['text'])

In [ ]:
# Load knowledge base .txt files and split into paragraph-level chunks
kb_docs = []
for kb_file in sorted(KB_DIR.glob('*.txt')):
    text = kb_file.read_text(encoding='utf-8')
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    for i, para in enumerate(paragraphs):
        if len(para) < 50:  # skip very short lines (headers, separators)
            continue
        kb_docs.append({
            'id': f"kb_{kb_file.stem}_{i:03d}",
            'text': para,
            'metadata': {
                'type': 'knowledge_base',
                'source': kb_file.stem,
            }
        })

print(f'Loaded {len(kb_docs)} KB chunks from {len(list(KB_DIR.glob("*.txt")))} files')

# Merge all document types
all_documents = doctor_docs + specialty_docs + kb_docs
print(f'\nTotal RAG documents: {len(all_documents)}')
print(f'Doctors: {len(doctor_docs)}')
print(f'Specialties: {len(specialty_docs)}')
print(f'KB chunks: {len(kb_docs)}')

In [ ]:
# Save processed documents to disk
processed_path = DATA_DIR / 'processed' / 'rag_documents.json'
with open(processed_path, 'w', encoding='utf-8') as f:
    json.dump(all_documents, f, ensure_ascii=False, indent=2)

print(f'Saved to: {processed_path}')

# Character length stats per document type
chars = [len(d['text']) for d in all_documents]
df_chars = pd.DataFrame({
    'chars': chars,
    'type': [d['metadata']['type'] for d in all_documents]
})

print(f'\nAvg document length: {sum(chars)/len(chars):.0f} chars')
print(f'Min: {min(chars)} Max: {max(chars)}')
print()
print(df_chars.groupby('type')['chars'].agg(['mean','min','max']).round(0))

## Generate evaluation dataset

20 questions with golden answers used by `03_rag_evaluation.ipynb` to score retrieval quality.

In [ ]:
eval_dataset = [
    # --- Symptom routing ---
    {
        'id': 'eval_001',
        'category': 'symptom_routing',
        'question': 'Имам болки в гърдите и задух. При кой специалист да отида?',
        'ground_truth': 'При кардиолог. Болка в гърдите и задух са симптоми на сърдечно-съдови заболявания.',
        'expected_specialty': 'кардиолог',
        'urgency': 'high'
    },
    {
        'id': 'eval_002',
        'category': 'symptom_routing',
        'question': 'Имам силно главоболие от 3 дни и изтръпване в дясната ръка.',
        'ground_truth': 'При невролог. Главоболие с изтръпване на крайник изисква неврологичен преглед.',
        'expected_specialty': 'невролог',
        'urgency': 'high'
    },
    {
        'id': 'eval_003',
        'category': 'symptom_routing',
        'question': 'Коляното ме боли много след бягане, има лек оток.',
        'ground_truth': 'При ортопед. Болка и оток в колято след спорт изискват ортопедичен преглед.',
        'expected_specialty': 'ортопед',
        'urgency': 'low'
    },
    {
        'id': 'eval_004',
        'category': 'symptom_routing',
        'question': 'Имам обрив и сърбеж по ръцете от 2 седмици.',
        'ground_truth': 'При дерматолог. Хроничен обрив и сърбеж изискват дерматологична консултация.',
        'expected_specialty': 'дерматолог',
        'urgency': 'low'
    },
    {
        'id': 'eval_005',
        'category': 'symptom_routing',
        'question': 'Имам киселини и болка в стомаха след хранене.',
        'ground_truth': 'При гастроентеролог. Киселини и стомашни болки след хранене са симптоми на рефлукс или гастрит.',
        'expected_specialty': 'гастроентеролог',
        'urgency': 'low'
    },
    {
        'id': 'eval_006',
        'category': 'symptom_routing',
        'question': 'Напоследък се уморявам много бързо, качих 5 кг без причина и ми е студено.',
        'ground_truth': 'При ендокринолог. Умора, наддаване на тегло и непоносимост към студ са симптоми на хипотиреоидизъм.',
        'expected_specialty': 'ендокринолог',
        'urgency': 'medium'
    },
    {
        'id': 'eval_007',
        'category': 'symptom_routing',
        'question': 'Имам хронична кашлица и задух при стълбите.',
        'ground_truth': 'При пулмолог. Хронична кашлица и задух при усилие изискват белодробен преглед.',
        'expected_specialty': 'пулмолог',
        'urgency': 'medium'
    },
    {
        'id': 'eval_008',
        'category': 'symptom_routing',
        'question': 'Ухото ме боли и чувам по-зле от вчера.',
        'ground_truth': 'При УНГ лекар. Болка в ухото и намален слух изискват отоскопия.',
        'expected_specialty': 'УНГ лекар',
        'urgency': 'medium'
    },
    {
        'id': 'eval_009',
        'category': 'symptom_routing',
        'question': 'Имам замъглено зрение от няколко дни.',
        'ground_truth': 'При офталмолог. Замъглено зрение изисква очен преглед.',
        'expected_specialty': 'офталмолог',
        'urgency': 'medium'
    },
    {
        'id': 'eval_010',
        'category': 'symptom_routing',
        'question': 'Чувствам се много тъжен от месеци, не мога да спя и нямам желание за нищо.',
        'ground_truth': 'При психиатър. Продължителна тъга, безсъние и апатия са симптоми на депресия.',
        'expected_specialty': 'психиатър',
        'urgency': 'medium'
    },
    # --- Doctor info ---
    {
        'id': 'eval_011',
        'category': 'doctor_info',
        'question': 'Кои кардиолози работят в центъра?',
        'ground_truth': 'В центъра работят д-р Мария Иванова и д-р Георги Петров — и двамата кардиолози.',
        'expected_specialty': 'кардиолог',
        'urgency': 'low'
    },
    {
        'id': 'eval_012',
        'category': 'doctor_info',
        'question': 'Кой невролог има най-много опит?',
        'ground_truth': 'Д-р Елена Николова с 22 години опит и докторска степен по неврология.',
        'expected_specialty': 'невролог',
        'urgency': 'low'
    },
    {
        'id': 'eval_013',
        'category': 'doctor_info',
        'question': 'Кой лекар приема по НЗОК за ортопедия?',
        'ground_truth': 'Д-р Ива Маринова приема пациенти по НЗОК с направление.',
        'expected_specialty': 'ортопед',
        'urgency': 'low'
    },
    {
        'id': 'eval_014',
        'category': 'doctor_info',
        'question': 'Колко струва преглед при гастроентеролог?',
        'ground_truth': 'Прегледът при гастроентеролог струва 80–100 лева.',
        'expected_specialty': 'гастроентеролог',
        'urgency': 'low'
    },
    # --- Center info ---
    {
        'id': 'eval_015',
        'category': 'center_info',
        'question': 'Как работи центърът — до колко часа?',
        'ground_truth': 'Центърът работи Понеделник-Петък от 08:00 до 19:00, в събота от 09:00 до 14:00.',
        'expected_specialty': None,
        'urgency': 'low'
    },
    {
        'id': 'eval_016',
        'category': 'center_info',
        'question': 'Имате ли паркинг?',
        'ground_truth': 'Да, има платен паркинг пред сградата на цена 2 лева/час.',
        'expected_specialty': None,
        'urgency': 'low'
    },
    {
        'id': 'eval_017',
        'category': 'center_info',
        'question': 'Трябва ли ми направление за преглед?',
        'ground_truth': 'Не е задължително. Можете да запишете час директно. Направление е нужно само ако искате да ползвате НЗОК.',
        'expected_specialty': None,
        'urgency': 'low'
    },
    # --- Urgency detection ---
    {
        'id': 'eval_018',
        'category': 'urgency',
        'question': 'Имам остра болка в гърдите и изпотяване.',
        'ground_truth': 'Това са потенциални симптоми на сърдечен удар. Незабавно се обадете на 112 или отидете в Спешно отделение.',
        'expected_specialty': None,
        'urgency': 'emergency'
    },
    {
        'id': 'eval_019',
        'category': 'urgency',
        'question': 'Внезапно не мога да говоря добре и едната ми ръка е слаба.',
        'ground_truth': 'Това са симптоми на инсулт. Незабавно се обадете на 112.',
        'expected_specialty': None,
        'urgency': 'emergency'
    },
    # --- Ambiguous / multi-symptom ---
    {
        'id': 'eval_020',
        'category': 'ambiguous',
        'question': 'Много се умирявам напоследък.',
        'ground_truth': 'Умората може да е симптом на различни заболявания. Нужна е повече информация — има ли и други симптоми?',
        'expected_specialty': None,
        'urgency': 'low'
    },
]

eval_path = DATA_DIR / 'processed' / 'eval_dataset.json'
with open(eval_path, 'w', encoding='utf-8') as f:
    json.dump(eval_dataset, f, ensure_ascii=False, indent=2)

df_eval = pd.DataFrame(eval_dataset)
print(f'Evaluation dataset saved: {len(eval_dataset)} questions')
print()
print('By category:')
print(df_eval['category'].value_counts().to_string())
print()
print('By urgency:')
print(df_eval['urgency'].value_counts().to_string())

## Summary

In [ ]:
print('=' * 50)
print('DATASET SUMMARY')
print('=' * 50)
print(f'Doctors: {len(doctors)}')
print(f'Specialties: {len(specialties)}')
print(f'Appointment slots: {sum(len(v["slots"]) for v in appointments.values())}')
print(f'RAG documents: {len(all_documents)}')
print(f'Eval questions: {len(eval_dataset)}')
print()
print('Files in data/processed/')
for p in sorted((DATA_DIR / 'processed').rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'{str(p.relative_to(DATA_DIR/"processed")):<40} {size_kb:.1f} KB')
print()
print('Next step: 02_rag_indexing.ipynb')